1. Configuración inicia


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler


Cargar df_final_integrado.xlsx y verificar estructura

In [5]:
df = pd.read_excel(r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\Datos\df_final_integrado.xlsx")
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25034 entries, 0 to 25033
Data columns (total 19 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0       @Name( )                                     25034 non-null  object 
 1         Date                                       25034 non-null  object 
 2   PRUEBA DE PRODUCCION PETRÓLEO A 24 HORAS bbl/d   25034 non-null  float64
 3   PRUEBA DE PRODUCCIÓN AGUA A 24 HORAS bbl/d       25034 non-null  float64
 4       Prueba_pozo.Oil_24 + Prueba_pozo.Water_24    25034 non-null  float64
 5   PRUEBA DE PRODUCCIÓN GAS A 24 HORAS Mcf/d        25034 non-null  float64
 6     BSW   %                                        25028 non-null  float64
 7   GRAVEDAD API DEL PETROLEO                        24951 non-null  float64
 8   SALINIDAD   PPM PU                               21233 non-null  float64
 9   PRESION DE INTAKE psi       

,PRUEBA DE PRODUCCION PETRÓLEO A 24 HORAS bbl/d,PRUEBA DE PRODUCCIÓN AGUA A 24 HORAS bbl/d,Prueba_pozo.Oil_24 + Prueba_pozo.Water_24,PRUEBA DE PRODUCCIÓN GAS A 24 HORAS Mcf/d,BSW %,GRAVEDAD API DEL PETROLEO,SALINIDAD PPM PU,PRESION DE INTAKE psi,FRECUENCIA BOMBA Hz,AMPERAJE BOMBA Amp,PRESION DE TUBING psi,PRESION DE CASING psi,TEMPERATURA DE LA BOMBA Deg. F,ETAPAS DE LA BOMBA
count,25034.000000,25034.000000,25034.000000,25034.000000,25028.000000,24951.000000,21233.000000,16730.000000,20640.000000,20437.000000,24996.000000,7181.000000,14490.000000,21670.000000
mean,354.889866,975.634710,1330.524576,67.026387,59.972295,23.979750,15587.132615,1110.637770,62.212201,39.183158,290.962046,56.372650,270.091877,314.420074
std,276.888482,883.744166,882.155010,95.448335,33.180432,3.720506,18728.522498,855.202319,22.056968,16.960842,729.971322,49.323983,35.104771,226.297095
min,11.000000,0.600000,33.800000,0.000000,0.350000,16.400000,17.200000,16.050000,35.000000,8.000000,-14.700000,1.000000,32.000000,0.000000
25%,173.000000,222.000000,661.000000,15.000000,28.020000,20.700000,2500.000000,475.000000,51.000000,27.000000,70.000000,20.000000,255.000000,184.000000
50%,273.000000,845.000000,1177.000000,43.350000,75.970000,24.400000,9500.000000,790.030000,57.150000,37.000000,105.000000,50.000000,272.000000,300.000000
75%,424.000000,1397.000000,1733.000000,75.640000,87.970000,26.800000,17000.000000,1686.000000,62.000000,47.000000,180.000000,90.000000,292.000000,402.000000
max,2058.000000,4188.000000,4770.000000,1235.000000,99.010000,31.000000,85800.000000,26664.000000,178.000000,147.000000,3920.000000,370.000000,376.000000,1614.000000


Normalizar columnas 

In [7]:
# Normalizar nombres de columnas (sin espacios ni acentos)
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('Á', 'A').str.replace('É', 'E').str.replace('Í', 'I').str.replace('Ó', 'O').str.replace('Ú', 'U')

# Convertir fecha
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Verificar duplicados
duplicados=df.duplicated(subset=['POZO', 'Date']).sum()
duplicados

np.int64(8873)

Ver duplicados por pozo y fecha

In [8]:
# Filtrar solo los duplicados (manteniendo todos los registros que tienen al menos una duplicación)
df_duplicados = df[df.duplicated(subset=['POZO', 'Date'], keep=False)]

# Ordenar para facilitar la revisión
df_duplicados = df_duplicados.sort_values(by=['POZO', 'Date'])

# Ver los primeros casos
df_duplicados.head(20)


,@Name(_),Date,PRUEBA_DE_PRODUCCION_PETROLEO_A_24_HORAS_bbl/d,PRUEBA_DE_PRODUCCION_AGUA_A_24_HORAS_bbl/d,Prueba_pozo.Oil_24_+_Prueba_pozo.Water_24,PRUEBA_DE_PRODUCCION_GAS_A_24_HORAS_Mcf/d,BSW___%,GRAVEDAD_API_DEL_PETROLEO,SALINIDAD___PPM_PU,PRESION_DE_INTAKE_psi,FRECUENCIA_BOMBA_Hz,AMPERAJE_BOMBA_Amp,PRESION_DE_TUBING_psi,PRESION_DE_CASING_psi,TIPO_DE_BOMBA,TEMPERATURA_DE_LA_BOMBA_Deg._F,ETAPAS_DE_LA_BOMBA,POZO,CAMPO
4729,SCH-050UI,2015-01-04,432.0,1727.0,2159.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13602,SCH-050UI,2015-01-04,432.0,1727.0,2159.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4782,SCH-050UI,2015-01-12,426.0,1706.0,2132.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13655,SCH-050UI,2015-01-12,426.0,1706.0,2132.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4716,SCH-050UI,2015-01-26,435.0,1742.0,2177.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13589,SCH-050UI,2015-01-26,435.0,1742.0,2177.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4717,SCH-050UI,2015-01-30,423.0,1695.0,2118.0,59.0,80.03,24.3,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13590,SCH-050UI,2015-01-30,423.0,1695.0,2118.0,59.0,80.03,24.3,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4741,SCH-050UI,2015-02-06,431.0,1723.0,2154.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13614,SCH-050UI,2015-02-06,431.0,1723.0,2154.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1


Cuántos duplicados por pozo tienes

In [9]:
df_duplicados['POZO'].value_counts().head(10)


POZO
SCH-050UI         1340
SCH-127HS         1296
SCH-083HI         1198
SCH-106HS         1192
SCH-124UI         1148
SCH-186UI          916
SCHAO-473HS1UI     894
SCHAO-477HUI       864
SCHV-227HS         830
SCH-128HS          820
Name: count, dtype: int64

Revisar si esos duplicados tienen diferencias en variables operativas:

In [10]:
pozo = 'SCH-050UI'
df_duplicados[df_duplicados['POZO'] == pozo].sort_values('Date').head(10)


,@Name(_),Date,PRUEBA_DE_PRODUCCION_PETROLEO_A_24_HORAS_bbl/d,PRUEBA_DE_PRODUCCION_AGUA_A_24_HORAS_bbl/d,Prueba_pozo.Oil_24_+_Prueba_pozo.Water_24,PRUEBA_DE_PRODUCCION_GAS_A_24_HORAS_Mcf/d,BSW___%,GRAVEDAD_API_DEL_PETROLEO,SALINIDAD___PPM_PU,PRESION_DE_INTAKE_psi,FRECUENCIA_BOMBA_Hz,AMPERAJE_BOMBA_Amp,PRESION_DE_TUBING_psi,PRESION_DE_CASING_psi,TIPO_DE_BOMBA,TEMPERATURA_DE_LA_BOMBA_Deg._F,ETAPAS_DE_LA_BOMBA,POZO,CAMPO
4729,SCH-050UI,2015-01-04,432.0,1727.0,2159.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13602,SCH-050UI,2015-01-04,432.0,1727.0,2159.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4782,SCH-050UI,2015-01-12,426.0,1706.0,2132.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13655,SCH-050UI,2015-01-12,426.0,1706.0,2132.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4716,SCH-050UI,2015-01-26,435.0,1742.0,2177.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13589,SCH-050UI,2015-01-26,435.0,1742.0,2177.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4717,SCH-050UI,2015-01-30,423.0,1695.0,2118.0,59.0,80.03,24.3,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13590,SCH-050UI,2015-01-30,423.0,1695.0,2118.0,59.0,80.03,24.3,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
13614,SCH-050UI,2015-02-06,431.0,1723.0,2154.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1
4741,SCH-050UI,2015-02-06,431.0,1723.0,2154.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1


Normalizar fecha por origen desde excel

In [11]:


# 1. Copia de seguridad
df['Date_raw'] = df['Date']

# 2. Convertir a datetime, tratando números tipo Excel
def convertir_fecha(valor):
    try:
        # Si es número (float o int), interpretarlo como fecha de Excel
        if isinstance(valor, (int, float)):
            return pd.to_datetime('1899-12-30') + pd.to_timedelta(valor, unit='D')
        else:
            # Si ya es texto tipo fecha
            return pd.to_datetime(valor, errors='coerce')
    except:
        return pd.NaT

df['Date'] = df['Date'].apply(convertir_fecha)

# 3. Verificar resultado
df[['Date_raw', 'Date']].head(10)


,Date_raw,Date
0,2015-02-01,2015-02-01
1,2015-10-01,2015-10-01
2,2015-02-18,2015-02-18
3,2015-02-23,2015-02-23
4,2015-02-25,2015-02-25
5,2015-06-04,2015-06-04
6,2015-04-15,2015-04-15
7,2015-04-21,2015-04-21
8,2015-04-27,2015-04-27
9,2015-05-05,2015-05-05


In [12]:
df['Date'].isna().sum()  # verifica si alguna no se pudo convertir
df['Date'].min(), df['Date'].max()  # rango temporal de tus datos


(Timestamp('2012-12-07 00:00:00'), Timestamp('2025-12-10 00:00:00'))

Verificar nuevos duplicados

In [13]:
# Volver a verificar duplicados tras normalizar fechas
nuevos_duplicados = df.duplicated(subset=['POZO', 'Date']).sum()
print("Duplicados actuales:", nuevos_duplicados)


Duplicados actuales: 8873


In [14]:
# Filtrar solo los duplicados (manteniendo ambos registros)
df_duplicados_actuales = df[df.duplicated(subset=['POZO', 'Date'], keep=False)]

# Ordenar para verlos claramente
df_duplicados_actuales = df_duplicados_actuales.sort_values(['POZO', 'Date'])

# Ver los primeros duplicados
df_duplicados_actuales.head(10)


,@Name(_),Date,PRUEBA_DE_PRODUCCION_PETROLEO_A_24_HORAS_bbl/d,PRUEBA_DE_PRODUCCION_AGUA_A_24_HORAS_bbl/d,Prueba_pozo.Oil_24_+_Prueba_pozo.Water_24,PRUEBA_DE_PRODUCCION_GAS_A_24_HORAS_Mcf/d,BSW___%,GRAVEDAD_API_DEL_PETROLEO,SALINIDAD___PPM_PU,PRESION_DE_INTAKE_psi,FRECUENCIA_BOMBA_Hz,AMPERAJE_BOMBA_Amp,PRESION_DE_TUBING_psi,PRESION_DE_CASING_psi,TIPO_DE_BOMBA,TEMPERATURA_DE_LA_BOMBA_Deg._F,ETAPAS_DE_LA_BOMBA,POZO,CAMPO,Date_raw
4729,SCH-050UI,2015-01-04,432.0,1727.0,2159.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-04
13602,SCH-050UI,2015-01-04,432.0,1727.0,2159.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-04
4782,SCH-050UI,2015-01-12,426.0,1706.0,2132.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-12
13655,SCH-050UI,2015-01-12,426.0,1706.0,2132.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,50.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-12
4716,SCH-050UI,2015-01-26,435.0,1742.0,2177.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-26
13589,SCH-050UI,2015-01-26,435.0,1742.0,2177.0,59.0,80.02,24.3,NaN,NaN,NaN,NaN,97.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-26
4717,SCH-050UI,2015-01-30,423.0,1695.0,2118.0,59.0,80.03,24.3,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-30
13590,SCH-050UI,2015-01-30,423.0,1695.0,2118.0,59.0,80.03,24.3,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-01-30
4741,SCH-050UI,2015-02-06,431.0,1723.0,2154.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-02-06
13614,SCH-050UI,2015-02-06,431.0,1723.0,2154.0,59.0,79.99,24.3,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,SCH-050UI,SACHA NORTE 1,2015-02-06


Eliminar duplicados exactos por pozo y fecha

In [15]:
# Eliminar duplicados manteniendo la primera ocurrencia
df = df.drop_duplicates(subset=['POZO', 'Date'], keep='first')

# Verificar que ya no existan duplicados
df.duplicated(subset=['POZO', 'Date']).sum()


np.int64(0)

In [16]:
print("Filas antes:", 25034)
print("Filas después:", len(df))


Filas antes: 25034
Filas después: 16161


In [19]:
df.to_excel(r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\Datos\df_final_limpio.xlsx", index=False)



